# Polished supplementary panels

Four figures over the 49 GTEx tissues, rebuilt from the exploratory notebooks so
they are reproducible on their own.

| Panel | Source | What |
|---|---|---|
| `sup_fig_UP_by_expression_polished` | `Fig1_Expression_vs_UP.ipynb`, `UP_splicing_by_exppression.All` | cumulative unproductive splicing by expression quintile, each tissue ranked by its own expression |
| `sup_fig_corr_across_tissues_polished` | same notebook, `UP_splicing_by_exppression.corr_across_tissues` | 49 x 49 rho: splicing in one tissue vs expression ordering from another |
| `sup_fig5A_polished` | `SupFig_GSEA.ipynb` | % unproductive junction reads vs *UPF3A* |
| `sup_fig5B_polished` | same | the same vs RNA integrity number |

**Each is written twice, once per correlation metric** — `_pearson` and
`_spearman` — so one can be chosen and applied consistently. Nothing else
differs between the two versions: the same points, curves and fits, only the
annotated rho, its P value, and (for the matrix) the values and clustering.

Two fixes carried over from the originals.

**The 49th panel keeps its axes.** `sup_fig5A` and `sup_fig5B` ended with

```python
ax[-1,-1].set_xticks([]); ax[-1,-1].set_yticks([])
ax[-1,-1].spines[['top','right','bottom','left']].set_visible(False)
```

which looks like tidying an unused corner. But 7 x 7 = 49 and there are exactly
49 tissues, so no panel is spare: that block was blanking the axes of the last
tissue's own panel, Whole Blood. It is not reproduced here.

**Statistics are printed in full** — rho, the exact P value and n on every
panel, rather than significance asterisks.

In [ ]:
import os
from matplotlib import pyplot as plt

import SupFig_Polished_helpers as H
import SupFig_Polished_plot_helpers as P

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

PLOTS_DIR = 'plots'
METRICS = ['pearson', 'spearman']
os.makedirs(PLOTS_DIR, exist_ok=True)
print('panels are written to', os.path.abspath(PLOTS_DIR))

In [ ]:
# Fast path: reload the plot-ready objects pickled by run_all().
data = H.load_plot_data('figure_data')

fig5A_data            = data['fig5A_data']
fig5B_data            = data['fig5B_data']
up_by_expression_data = data['up_by_expression_data']
corr_matrix_data      = data['corr_matrix_data']

print({k: (len(v) if isinstance(v, list) else 'matrix') for k, v in data.items()})

In [ ]:
# Full rebuild. Slow: one junction table per tissue for the quintile grid, the
# per-sample unproductive percentages for the two scatter grids, and 49 x 49
# tissue pairs for the correlation matrix. Run once, then use the cell above.

# data = H.run_all('figure_data')

### `sup_fig_UP_by_expression_polished`

For each tissue, genes are split into quintiles by that tissue's **own** median
expression, and the cumulative distribution of unproductive-splicing percentage
is drawn per quintile. This is `UP_splicing_by_exppression.All` from
`Fig1_Expression_vs_UP.ipynb`, with rho, P and n added to every panel.

Curves run pale to dark within each tissue's GTEx colour: palest is the lowest
expression quintile. Lowly expressed genes carry more unproductive splicing, so
the pale curves sit to the right.

In [ ]:
for metric in METRICS:
    P.plot_selfsorted_grid(up_by_expression_data, metric=metric)
    P.save_panel(f'sup_fig_UP_by_expression_polished_{metric}', PLOTS_DIR)
    plt.close('all')

### `sup_fig_corr_across_tissues_polished`

rho between unproductive splicing in one tissue (rows) and the gene expression
ordering of another (columns). The diagonal is the self-sorted case shown in the
grid above. Rows and columns are in Ward order, computed separately for each
metric, with GTEx tissue colours on both margins.

In [ ]:
for metric in METRICS:
    P.plot_corr_matrix(corr_matrix_data, metric=metric)
    P.save_panel(f'sup_fig_corr_across_tissues_polished_{metric}', PLOTS_DIR)
    plt.close('all')

### `sup_fig5A_polished`

Per-sample percentage of unproductive junction reads against *UPF3A* expression,
one panel per tissue. Dashed line is a robust linear fit (`statsmodels` RLM);
black marks Artery - Tibial, as in the original.

In [ ]:
for metric in METRICS:
    P.plot_scatter_grid(fig5A_data, xlabel='logUPF3A (TPM)', metric=metric)
    P.save_panel(f'sup_fig5A_polished_{metric}', PLOTS_DIR)
    plt.close('all')

### `sup_fig5B_polished`

The same against RNA integrity number, to show the *UPF3A* association is not a
degradation artefact.

In [ ]:
for metric in METRICS:
    P.plot_scatter_grid(fig5B_data, xlabel='RNA Integrity Number', metric=metric)
    P.save_panel(f'sup_fig5B_polished_{metric}', PLOTS_DIR)
    plt.close('all')